# BPE Tokenizer — Train on TinyStories

This notebook trains a Byte-Pair Encoding tokenizer from scratch on the
[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) dataset.

**Sections**
1. Install dependencies
2. Download & prepare the TinyStories corpus
3. BPE trainer (`train_bpe_parallel`)
4. BPE tokenizer (`BPETokenizer`)
5. Train
6. Save / load
7. Encode & decode examples
8. Vocabulary explorer


## 1 · Install dependencies

In [ ]:
# regex is a drop-in re replacement with Unicode property escapes (\p{L} etc.)
# datasets gives us the HuggingFace TinyStories loader
!pip install -q regex datasets tqdm

## 2 · Download & prepare TinyStories

We stream the training split and write a plain UTF-8 text file that the
trainer reads from disk. A `max_stories` cap lets you do a quick smoke-test
run without waiting for the full ~2 M stories.

In [ ]:
from datasets import load_dataset
from pathlib import Path
from tqdm.auto import tqdm

# ── config ────────────────────────────────────────────────────────────
CORPUS_PATH  = Path("/content/tinystories_train.txt")
MAX_STORIES  = None   # set to None to use the full ~2.1 M stories
STORY_SEP    = "<|endoftext|>"  # special token between stories
# ──────────────────────────────────────────────────────────────────────

if CORPUS_PATH.exists():
    print(f"Corpus already exists at {CORPUS_PATH}, skipping download.")
else:
    print("Streaming TinyStories from HuggingFace …")
    ds = load_dataset(
        "roneneldan/TinyStories",
        split="train",
        streaming=True,
    )

    written = 0
    with CORPUS_PATH.open("w", encoding="utf-8") as f:
        for example in tqdm(ds, total=MAX_STORIES, desc="Writing stories"):
            f.write(example["text"].strip())
            f.write("\n" + STORY_SEP + "\n")
            written += 1
            if MAX_STORIES and written >= MAX_STORIES:
                break

    size_mb = CORPUS_PATH.stat().st_size / 1024 / 1024
    print(f"\nWrote {written:,} stories → {CORPUS_PATH}  ({size_mb:.1f} MB)")

Streaming TinyStories from HuggingFace …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Writing stories: 0it [00:00, ?it/s]


Wrote 2,119,719 stories → /content/tinystories_train.txt  (1845.1 MB)


## 3 · BPE trainer

Key ideas:
- **Incremental pair-count updates** (delta patching) instead of full recount each merge → O(N + M · avg_affected) instead of O(N · M)
- **Max-heap** with lazy deletion for O(log M) best-pair lookup
- **Doubly-linked list** per pre-token for O(1) neighbour lookup during merge
- **Multiprocessing** for pre-tokenisation on large corpora (auto-disabled below 5 MB)

In [ ]:
"""
Optimized BPE trainer.
"""
from __future__ import annotations

import heapq
from collections import Counter, defaultdict
from dataclasses import dataclass
from multiprocessing import Pool, cpu_count
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import regex as re
from tqdm.auto import tqdm


PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
_PAT_RE = re.compile(PAT)
_MP_THRESHOLD_BYTES = 5 * 1024 * 1024


def _build_special_split_pattern(special_tokens):
    if not special_tokens:
        return None
    escaped = sorted((re.escape(t) for t in special_tokens), key=len, reverse=True)
    return re.compile("(" + "|".join(escaped) + ")")


def _split_on_special_tokens(text, pattern, special_set):
    if pattern is None:
        return [(text, False)]
    out = []
    for part in pattern.split(text):
        if part:
            out.append((part, part in special_set))
    return out


def _find_safe_chunk_ranges(text, boundary_token, target_num_chunks):
    n = len(text)
    if n == 0 or target_num_chunks <= 1 or boundary_token not in text:
        return [(0, n)]
    approx = max(1, n // target_num_chunks)
    starts = [0]
    probe = approx
    while probe < n:
        nb = text.find(boundary_token, probe)
        if nb == -1:
            break
        starts.append(nb)
        probe = nb + len(boundary_token) + approx
    starts = sorted(set(starts))
    ranges = []
    for i, s in enumerate(starts):
        e = starts[i + 1] if i + 1 < len(starts) else n
        if s < e:
            ranges.append((s, e))
    return ranges


@dataclass
class _ChunkJob:
    chunk_text: str
    special_tokens: List[str]


def _process_chunk(job):
    pattern = _build_special_split_pattern(job.special_tokens)
    special_set = frozenset(job.special_tokens)
    counts = Counter()
    for chunk, is_special in _split_on_special_tokens(job.chunk_text, pattern, special_set):
        if is_special:
            continue
        for m in _PAT_RE.finditer(chunk):
            b = m.group(0).encode("utf-8")
            counts[tuple(bytes([x]) for x in b)] += 1
    return counts


@dataclass
class _LinkedSeq:
    vals: List
    prev: List[int]
    next: List[int]
    head: int
    tail: int


def _build_linked_seq(seq):
    n = len(seq)
    vals = [None] + list(seq) + [None]
    prev = list(range(-1, n + 1))
    nxt  = list(range(1, n + 3))
    nxt[n + 1] = n + 1
    return _LinkedSeq(vals=vals, prev=prev, next=nxt, head=0, tail=n + 1)


def _linked_pairs(ls):
    i = ls.next[ls.head]
    while ls.next[i] != ls.tail:
        j = ls.next[i]
        yield i, j
        i = j


def _merge_in_linked(ls, a, b, ab):
    deltas = []
    i = ls.next[ls.head]
    while ls.next[i] != ls.tail:
        j = ls.next[i]
        if ls.vals[i] == a and ls.vals[j] == b:
            li = ls.prev[i]
            rj = ls.next[j]
            if ls.vals[li] is not None:
                deltas.append(((ls.vals[li], a), -1))
            if ls.vals[rj] is not None:
                deltas.append(((b, ls.vals[rj]), -1))
            ls.vals[i] = ab
            ls.next[i] = rj
            ls.prev[rj] = i
            if ls.vals[li] is not None:
                deltas.append(((ls.vals[li], ab), +1))
            if ls.vals[rj] is not None:
                deltas.append(((ab, ls.vals[rj]), +1))
        else:
            i = j
            continue
        i = ls.next[i]
    return deltas


class _PairHeap:
    def __init__(self, pair_counts):
        self._counts = dict(pair_counts)
        self._heap = [(-cnt, pair) for pair, cnt in self._counts.items()]
        heapq.heapify(self._heap)

    def update(self, pair, delta):
        if delta == 0:
            return
        old = self._counts.get(pair, 0)
        new = old + delta
        if new <= 0:
            self._counts.pop(pair, None)
        else:
            self._counts[pair] = new
            heapq.heappush(self._heap, (-new, pair))

    def best(self):
        while self._heap:
            neg_cnt, pair = self._heap[0]
            cnt = -neg_cnt
            actual = self._counts.get(pair, 0)
            if actual != cnt:
                heapq.heappop(self._heap)
                continue
            return pair, cnt
        return None

    def remove_pair(self, pair):
        self._counts.pop(pair, None)


def train_bpe_parallel(
    input_path,
    vocab_size,
    special_tokens=None,
    num_workers=None,
    show_progress=True,
    log_every=100,
):
    if special_tokens is None:
        special_tokens = []
    min_vocab = 256 + len(special_tokens)
    if vocab_size < min_vocab:
        raise ValueError(f"vocab_size={vocab_size} is too small; minimum is {min_vocab}")

    text = Path(input_path).read_text(encoding="utf-8")

    vocab = {i: bytes([i]) for i in range(256)}
    next_token_id = 256
    for tok in special_tokens:
        vocab[next_token_id] = tok.encode("utf-8")
        next_token_id += 1

    if num_workers is None:
        num_workers = max(1, cpu_count() - 1)

    use_mp = len(text.encode()) >= _MP_THRESHOLD_BYTES and num_workers > 1

    if special_tokens:
        ranges = _find_safe_chunk_ranges(text, special_tokens[0], num_workers if use_mp else 1)
    else:
        ranges = [(0, len(text))]

    jobs = [_ChunkJob(chunk_text=text[s:e], special_tokens=special_tokens) for s, e in ranges]

    pretoken_counter = Counter()
    if use_mp and len(jobs) > 1:
        with Pool(processes=num_workers) as pool:
            it = pool.imap_unordered(_process_chunk, jobs, chunksize=1)
            if show_progress:
                it = tqdm(it, total=len(jobs), desc="Pre-tokenising", unit="chunk")
            for partial in it:
                pretoken_counter.update(partial)
    else:
        it2 = tqdm(jobs, desc="Pre-tokenising", unit="chunk") if show_progress else jobs
        for job in it2:
            pretoken_counter.update(_process_chunk(job))

    pretoken_seqs = {
        seq: (_build_linked_seq(seq), freq)
        for seq, freq in pretoken_counter.items()
    }

    pair_counts = defaultdict(int)
    for seq_key, (ls, freq) in pretoken_seqs.items():
        for i, j in _linked_pairs(ls):
            pair_counts[(ls.vals[i], ls.vals[j])] += freq

    heap = _PairHeap(pair_counts)
    merges = []
    target = vocab_size - len(vocab)
    merge_bar = tqdm(total=target, desc="Learning BPE merges", unit="merge") if show_progress else None

    while len(vocab) < vocab_size:
        best = heap.best()
        if best is None:
            break
        pair, best_count = best
        a, b = pair
        ab = a + b
        heap.remove_pair(pair)
        vocab[next_token_id] = ab
        merges.append(pair)
        next_token_id += 1

        agg_deltas = defaultdict(int)
        for seq_key, (ls, freq) in pretoken_seqs.items():
            for delta_pair, delta_val in _merge_in_linked(ls, a, b, ab):
                agg_deltas[delta_pair] += delta_val * freq
        agg_deltas[pair] -= best_count
        for dp, dv in agg_deltas.items():
            if dv:
                heap.update(dp, dv)

        if merge_bar is not None:
            merge_bar.update(1)
            if len(merges) % log_every == 0 or len(vocab) == vocab_size:
                merge_bar.set_postfix(
                    vocab_size=len(vocab),
                    last_merge=(a + b"+" + b)[:20],
                    count=best_count,
                )

    if merge_bar is not None:
        merge_bar.close()
    return vocab, merges


print("Trainer loaded ✓")

Trainer loaded ✓


## 4 · BPETokenizer

Key optimisations over a naive implementation:
- `_apply_merges_to_pretoken`: **O(n log n)** priority-queue instead of O(n²) full-scan
- `encode`: per-instance **LRU cache** (65 536 entries) — repeated words cost one dict lookup
- Special-token split pattern compiled **once** at construction

In [ ]:
import heapq
from functools import lru_cache
from typing import Dict, Iterable, Iterator, List, Optional, Tuple


class BPETokenizer:
    """A Byte-Pair Encoding (BPE) tokenizer.

    This tokenizer implements BPE to convert text into a sequence of token IDs
    and vice-versa. It supports custom vocabularies, merge rules, and special tokens.
    Key features include efficient merging using a priority queue and caching
    for pre-token encoding.

    Args:
        vocab (Dict[int, bytes]): A dictionary mapping token IDs to their byte representations.
        merges (List[Tuple[bytes, bytes]]): A list of byte pair merge rules in order of application.
        special_tokens (Optional[List[str]]): A list of string representations of special tokens.
    """
    def __init__(self, vocab, merges, special_tokens=None):
        self.vocab = dict(vocab)
        self.merges = list(merges)
        self.special_tokens = special_tokens or []

        self.token_to_id = {tok_bytes: tok_id for tok_id, tok_bytes in self.vocab.items()}

        next_id = max(self.vocab.keys(), default=-1) + 1
        for tok in self.special_tokens:
            tok_bytes = tok.encode("utf-8")
            if tok_bytes not in self.token_to_id:
                self.vocab[next_id] = tok_bytes
                self.token_to_id[tok_bytes] = next_id
                next_id += 1

        self.merge_ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self._split_pattern = _build_special_split_pattern(self.special_tokens)
        self._special_set = frozenset(self.special_tokens)

        @lru_cache(maxsize=1 << 16)
        def _cached_encode_pretoken(pretok: str) -> Tuple[int, ...]:
            pieces = self._apply_merges_to_pretoken(pretok)
            return tuple(self.token_to_id[p] for p in pieces)
        self._cached_encode_pretoken = _cached_encode_pretoken

    @classmethod
    def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None):
        """Loads a tokenizer from vocabulary and merge files.

        Args:
            vocab_filepath (str): Path to the vocabulary file.
            merges_filepath (str): Path to the merge rules file.
            special_tokens (Optional[List[str]]): A list of string representations of special tokens.

        Returns:
            BPETokenizer: An initialized BPETokenizer instance.
        """
        vocab = {}
        with open(vocab_filepath, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                tok_id_str, hex_bytes = line.split("\t", 1)
                vocab[int(tok_id_str)] = bytes.fromhex(hex_bytes)
        merges = []
        with open(merges_filepath, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                left_hex, right_hex = line.split("\t", 1)
                merges.append((bytes.fromhex(left_hex), bytes.fromhex(right_hex)))
        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)

    def save(self, vocab_filepath, merges_filepath):
        """Saves the tokenizer's vocabulary and merge rules to files.

        Args:
            vocab_filepath (str): Path where the vocabulary file will be saved.
            merges_filepath (str): Path where the merge rules file will be saved.
        """
        with open(vocab_filepath, "w", encoding="utf-8") as f:
            for tok_id in sorted(self.vocab):
                f.write(f"{tok_id}\t{self.vocab[tok_id].hex()}\n")
        with open(merges_filepath, "w", encoding="utf-8") as f:
            for left, right in self.merges:
                f.write(f"{left.hex()}\t{right.hex()}\n")

    def _apply_merges_to_pretoken(self, pretok: str) -> List[bytes]:
        """Applies BPE merges to a single pre-token.

        This method takes a pre-token (e.g., a word or punctuation) and applies
        all learned BPE merges to it, resulting in a sequence of byte pieces.
        It uses a priority queue to apply merges efficiently based on their rank.

        Args:
            pretok (str): The pre-token string to be merged.

        Returns:
            List[bytes]: A list of byte pieces after applying merges.
        """
        seq = [bytes([b]) for b in pretok.encode("utf-8")]
        n = len(seq)
        if n == 1:
            return seq
        if n == 2:
            if (seq[0], seq[1]) in self.merge_ranks:
                return [seq[0] + seq[1]]
            return seq
        prev = list(range(-1, n))
        nxt  = list(range(1, n + 1))
        heap = []
        for i in range(n - 1):
            rank = self.merge_ranks.get((seq[i], seq[i + 1]))
            if rank is not None:
                heapq.heappush(heap, (rank, i))
        valid = set(range(n))
        while heap:
            rank, i = heapq.heappop(heap)
            if i not in valid:
                continue
            j = nxt[i]
            if j >= n or j not in valid:
                continue
            if self.merge_ranks.get((seq[i], seq[j])) != rank:
                continue
            seq[i] = seq[i] + seq[j]
            valid.discard(j)
            nxt[i] = nxt[j]
            if nxt[j] < n:
                prev[nxt[j]] = i
            li = prev[i]
            if li >= 0 and li in valid:
                r = self.merge_ranks.get((seq[li], seq[i]))
                if r is not None:
                    heapq.heappush(heap, (r, li))
            ri = nxt[i]
            if ri < n and ri in valid:
                r = self.merge_ranks.get((seq[i], seq[ri]))
                if r is not None:
                    heapq.heappush(heap, (r, i))
        return [seq[i] for i in range(n) if i in valid]

    def encode(self, text: str) -> List[int]:
        """Encodes a given text string into a sequence of token IDs.

        This method first splits the text based on special tokens, then pre-tokenizes
        the non-special parts, and finally applies BPE merges to convert them into token IDs.

        Args:
            text (str): The input text string to encode.

        Returns:
            List[int]: A list of token IDs representing the encoded text.
        """
        ids = []
        for chunk, is_special in _split_on_special_tokens(
            text, self._split_pattern, self._special_set
        ):
            if is_special:
                ids.append(self.token_to_id[chunk.encode("utf-8")])
                continue
            for m in _PAT_RE.finditer(chunk):
                ids.extend(self._cached_encode_pretoken(m.group(0)))
        return ids

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        """Encodes an iterable of text strings into a stream of token IDs.

        Args:
            iterable (Iterable[str]): An iterable yielding text strings.

        Yields:
            int: The next token ID.
        """
        for chunk in iterable:
            yield from self.encode(chunk)

    def decode(self, ids: List[int]) -> str:
        """Decodes a sequence of token IDs back into a text string.

        Args:
            ids (List[int]): A list of token IDs to decode.

        Returns:
            str: The decoded text string.
        """
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")


print("BPETokenizer loaded ✓")

BPETokenizer loaded ✓


## 5 · Train

Adjust `VOCAB_SIZE` to taste. Common choices:
| Size | Character | Use case |
|------|-----------|----------|
| 1 000 | tiny | quick smoke-test |
| 4 096 | small | toy LM experiments |
| 10 000 | medium | TinyStories-scale models |
| 32 000 | large | GPT-2 / LLaMA-style |

> **Tip:** on a free Colab CPU, 500 k stories + vocab 4 096 finishes in ~5 min.
> For 32 k vocab on the full dataset, use a GPU runtime or reduce `MAX_STORIES`.

In [ ]:
VOCAB_SIZE     = 70000
SPECIAL_TOKENS = ["<|endoftext|>"]

import time
start = time.time()

vocab, merges = train_bpe_parallel(
    input_path=str(CORPUS_PATH),
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
)

elapsed = time.time() - start

# Build the tokenizer immediately so all subsequent cells can use it
tokenizer = BPETokenizer(vocab, merges, special_tokens=SPECIAL_TOKENS)

print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Vocab size : {len(tokenizer.vocab):,}")
print(f"Merges     : {len(tokenizer.merges):,}")

Pre-tokenising:   0%|          | 0/7 [00:02<?, ?chunk/s]

Learning BPE merges:   0%|          | 0/69743 [00:00<?, ?merge/s]


Training complete in 4791.0s
Vocab size : 70,000
Merges     : 69,743


## 5b · Training summary

What exactly came out of training? This cell breaks the vocabulary into its three layers and shows the first and last merge rules — two windows into what the algorithm learned.

In [ ]:
import time

# ── Token type breakdown ──────────────────────────────────────────────
byte_tokens    = sum(1 for tid in tokenizer.vocab if tid < 256)
special_count  = len(tokenizer.special_tokens)
merged_tokens  = len(tokenizer.vocab) - byte_tokens - special_count

print("Token type breakdown")
print("-" * 40)
print(f"  Byte tokens   (IDs   0–255) : {byte_tokens}")
print(f"  Special tokens              : {special_count}")
print(f"  Merged tokens (IDs 256+)    : {merged_tokens:,}")
print(f"  Total                       : {len(tokenizer.vocab):,}")

# Why this matters:
# The 256 byte tokens are the 'alphabet' — every possible input byte,
# so the tokenizer can never produce an <UNK> token.
# Every token above ID 255 is a merge rule made concrete.

print()

# ── Printable byte tokens ──────────────────────────────────────────────
print("Sample byte tokens (printable ASCII)")
print("-" * 40)
shown = 0
for i in range(256):
    ch = tokenizer.vocab[i].decode("utf-8", errors="replace")
    if ch.isprintable() and ch.strip():
        print(f"  ID {i:3d}  →  {ch!r}")
        shown += 1
        if shown == 12:
            break

print()

# ── Special tokens ─────────────────────────────────────────────────────
print("Special tokens")
print("-" * 40)
for tok_str in tokenizer.special_tokens:
    tok_bytes = tok_str.encode("utf-8")
    tok_id = tokenizer.token_to_id[tok_bytes]
    print(f"  ID {tok_id}  →  {tok_str!r}")

print()

Token type breakdown
----------------------------------------
  Byte tokens   (IDs   0–255) : 256
  Special tokens              : 1
  Merged tokens (IDs 256+)    : 69,743
  Total                       : 70,000

Sample byte tokens (printable ASCII)
----------------------------------------
  ID  33  →  '!'
  ID  34  →  '"'
  ID  35  →  '#'
  ID  36  →  '$'
  ID  37  →  '%'
  ID  38  →  '&'
  ID  39  →  "'"
  ID  40  →  '('
  ID  41  →  ')'
  ID  42  →  '*'
  ID  43  →  '+'
  ID  44  →  ','

Special tokens
----------------------------------------
  ID 256  →  '<|endoftext|>'



### First and last merge rules

The **first merges** are always the most frequent byte pairs in the raw corpus — usually spaces glued to letters, or the most common English digrams.

The **last merges** are long, specific strings that only just crossed the frequency threshold — rare enough that they were the very last things worth merging.

In [ ]:
def show_merges(merges, vocab, label, indices):
    print(f"{label}")
    print("-" * 52)
    for i in indices:
        left, right = merges[i]
        merged = left + right
        l_str = left.decode("utf-8",  errors="replace")
        r_str = right.decode("utf-8", errors="replace")
        m_str = merged.decode("utf-8", errors="replace")
        # find the ID of the merged token
        merged_id = 256 + len(tokenizer.special_tokens) + i
        print(f"  Merge {i+1:>5d}  {l_str!r:>12} + {r_str!r:<12} →  {m_str!r}  (ID {merged_id})")
    print()

n = len(tokenizer.merges)
show_merges(tokenizer.merges, tokenizer.vocab, "First 10 merges  (most frequent byte pairs)", range(10))
show_merges(tokenizer.merges, tokenizer.vocab, "Last 10 merges   (rarest pairs still worth merging)", range(n-10, n))

# Reading the first merges tells you what the corpus is made of.
# Reading the last merges tells you where the frequency cliff is —
# everything below this count was not worth its own token.

First 10 merges  (most frequent byte pairs)
----------------------------------------------------
  Merge     1           'h' + 'e'          →  'he'  (ID 257)
  Merge     2           ' ' + 't'          →  ' t'  (ID 258)
  Merge     3           ' ' + 'a'          →  ' a'  (ID 259)
  Merge     4           ' ' + 's'          →  ' s'  (ID 260)
  Merge     5           ' ' + 'w'          →  ' w'  (ID 261)
  Merge     6           'n' + 'd'          →  'nd'  (ID 262)
  Merge     7          ' t' + 'he'         →  ' the'  (ID 263)
  Merge     8           'e' + 'd'          →  'ed'  (ID 264)
  Merge     9          ' a' + 'nd'         →  ' and'  (ID 265)
  Merge    10          ' t' + 'o'          →  ' to'  (ID 266)

Last 10 merges   (rarest pairs still worth merging)
----------------------------------------------------
  Merge 69734         '\n ' + '                ' →  '\n                 '  (ID 69990)
  Merge 69735  '\n                 ' + '      '     →  '\n                       '  (ID 69991)
 

### Compression ratio

The most direct measure of what BPE actually achieved: how many tokens does it take to represent the same text compared to raw bytes?

In [ ]:
test_story = (
    "Once upon a time, there was a little girl named Lily who loved to run "
    "through the forest collecting mushrooms and flowers. She was very happy."
)


raw_bytes   = len(test_story.encode("utf-8"))
token_ids   = tokenizer.encode(test_story)
num_tokens  = len(token_ids)
ratio       = raw_bytes / num_tokens

print("Compression on a sample story")
print("-" * 40)
print(f"  Raw bytes  : {raw_bytes}")
print(f"  Tokens     : {num_tokens}")
print(f"  Ratio      : {ratio:.2f} bytes / token")
print(f"  Compression: {(1 - num_tokens/raw_bytes)*100:.1f}% fewer symbols than raw bytes")
print()

# Show the actual token strings so you can see how the text was chunked
token_strings = [tokenizer.vocab[i].decode("utf-8", errors="replace") for i in token_ids]
print("Token boundaries  (| separates tokens):")
print("|" + "|".join(token_strings) + "|")

Compression on a sample story
----------------------------------------
  Raw bytes  : 142
  Tokens     : 29
  Ratio      : 4.90 bytes / token
  Compression: 79.6% fewer symbols than raw bytes

Token boundaries  (| separates tokens):
|Once| upon| a| time|,| there| was| a| little| girl| named| Lily| who| loved| to| run| through| the| forest| collecting| mushrooms| and| flowers|.| She| was| very| happy|.|


## 6 · Save & reload

Vocab is stored as `<id>\t<hex_bytes>` and merges as `<left_hex>\t<right_hex>`.
Both files are plain text — easy to inspect or version-control.

In [ ]:
VOCAB_PATH  = "/content/tinystories_bpe.vocab"
MERGES_PATH = "/content/tinystories_bpe.merges"

tokenizer = BPETokenizer(vocab, merges, special_tokens=SPECIAL_TOKENS)
tokenizer.save(VOCAB_PATH, MERGES_PATH)
print("Saved.")

# Round-trip check
tokenizer = BPETokenizer.from_files(VOCAB_PATH, MERGES_PATH, special_tokens=SPECIAL_TOKENS)
print(f"Reloaded tokenizer  vocab={len(tokenizer.vocab):,}  merges={len(tokenizer.merges):,}")

Saved.
Reloaded tokenizer  vocab=10,000  merges=9,743


## 7 · Encode & decode examples

In [ ]:
samples = [
    "Once upon a time, there was a little girl named Lily.",
    "She loved to play in the garden with her dog.",
    "<|endoftext|>",
    "Hello, world! 123",
]

for text in samples:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    tokens = [tokenizer.vocab[i] for i in ids]
    print(f"Input   : {text!r}")
    print(f"IDs     : {ids}")
    print(f"Tokens  : {[t.decode('utf-8', errors='replace') for t in tokens]}")
    print(f"Decoded : {decoded!r}")
    print()

Input   : 'Once upon a time, there was a little girl named Lily.'
IDs     : [431, 447, 259, 396, 44, 399, 282, 259, 397, 446, 500, 364, 46]
Tokens  : ['Once', ' upon', ' a', ' time', ',', ' there', ' was', ' a', ' little', ' girl', ' named', ' Lily', '.']
Decoded : 'Once upon a time, there was a little girl named Lily.'

Input   : 'She loved to play in the garden with her dog.'
IDs     : [1076, 503, 266, 359, 316, 263, 846, 342, 309, 634, 46]
Tokens  : ['She', ' loved', ' to', ' play', ' in', ' the', ' garden', ' with', ' her', ' dog', '.']
Decoded : 'She loved to play in the garden with her dog.'

Input   : '<|endoftext|>'
IDs     : [256]
Tokens  : ['<|endoftext|>']
Decoded : '<|endoftext|>'

Input   : 'Hello, world! 123'
IDs     : [1410, 44, 1281, 33, 5935, 50, 51]
Tokens  : ['Hello', ',', ' world', '!', ' 1', '2', '3']
Decoded : 'Hello, world! 123'



## 8 · Vocabulary explorer

Peek at what the tokenizer learned — sorted by token length so you can
see the longest merged tokens at the top.

In [ ]:
# Show the N longest tokens (most merges applied)
N = 40

sorted_vocab = sorted(
    tokenizer.vocab.items(),
    key=lambda kv: len(kv[1]),
    reverse=True,
)

print(f"{'ID':>6}  {'Bytes':>6}  Token")
print("-" * 40)
for tok_id, tok_bytes in sorted_vocab[:N]:
    display = tok_bytes.decode("utf-8", errors="replace")
    print(f"{tok_id:>6}  {len(tok_bytes):>6}  {display!r}")

    ID   Bytes  Token
----------------------------------------
  6449      15  ' accomplishment'
  8090      15  ' responsibility'
  9106      15  ' disappointment'
  9774      15  ' recommendation'
  3606      14  ' uncomfortable'
  3822      14  ' compassionate'
  4865      14  ' understanding'
  5700      14  ' neighbourhood'
  6249      14  ' Unfortunately'
  6423      14  ' determination'
  7648      14  ' granddaughter'
  7712      14  ' congratulated'
  7722      14  ' encouragement'
  7834      14  ' unfortunately'
  8530      14  ' extraordinary'
  9668      14  ' possibilities'
   256      13  '<|endoftext|>'
  2130      13  ' accidentally'
  2559      13  ' disappointed'
  3902      13  ' enthusiastic'
  4692      13  ' veterinarian'
  4895      13  ' strawberries'
  4930      13  ' refrigerator'
  5125      13  ' neighborhood'
  5732      13  ' instructions'
  6386      13  'Unfortunately'
  6544      13  ' consequences'
  6590      13  ' firefighters'
  6852      13  ' sur

In [ ]:
# Token-length histogram
from collections import Counter

length_counts = Counter(len(v) for v in tokenizer.vocab.values())
print("Token length distribution:")
print(f"{'Length':>8}  {'Count':>8}  Bar")
print("-" * 50)
for length in sorted(length_counts):
    count = length_counts[length]
    bar = "█" * min(count // 10, 50)
    print(f"{length:>8}  {count:>8}  {bar}")

Token length distribution:
  Length     Count  Bar
--------------------------------------------------
       1       256  █████████████████████████
       2       373  █████████████████████████████████████
       3       890  ██████████████████████████████████████████████████
       4      1409  ██████████████████████████████████████████████████
       5      1798  ██████████████████████████████████████████████████
       6      1629  ██████████████████████████████████████████████████
       7      1288  ██████████████████████████████████████████████████
       8      1056  ██████████████████████████████████████████████████
       9       625  ██████████████████████████████████████████████████
      10       355  ███████████████████████████████████
      11       194  ███████████████████
      12        82  ████████
      13        29  ██
      14        12  █
      15         4  
